In [ ]:
import os
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from astra.src.embedding import AstraEmbedding
from astra.src.preprocessing import contrastive_data_loader

### Data Loader for Contrastive Learning (Anchor, Positive, Negative)

In [ ]:
#
# This script demonstrates how to use the contrastive_data_loader function
#
n_views = 3
batch_size = 200
apply_white_noise=[False, True, True]
apply_binning=[False, False, True]
apply_outlier=[False, False, True]
maxlens=[{'g': 100, 'r': 200, 'i': 50}, {'g': 100, 'r': 150, 'i': 50}, {'g': 100, 'r': 200, 'i': 50}] 
bin_widths = [5, 5, 5]
drop_rates = [0.0, 0.0, 0.60]
noise_levels = [0.10, 0.0, 0.10] 
buffer_size = 10000  
build_seq_len=tf.cast(sum(maxlens[0].values()), tf.int32)  # Final fixed length for sequences 

path_to_read = "/media3/majumder/dataset/resampled_multi-class/val/"
#
loaders = contrastive_data_loader( source=path_to_read,
                                    n_views=n_views, 
                                    seed=np.random.randint(1024), 
                                    batch_size=batch_size,
                                    build_seq_len=build_seq_len,
                                    apply_white_noise=apply_white_noise,
                                    noise_levels=noise_levels,
                                    apply_binning=apply_binning,
                                    apply_outlier=apply_outlier,
                                    maxlens=maxlens,
                                    bin_widths=bin_widths,
                                    drop_rates=drop_rates,
                                    buffer_size=buffer_size
                                )

# im1, im2, im3 = next(iter(loaders))
# print(im1, im2, im3)
# for anchor, positive, negative in loaders:
#   print(anchor.keys(), anchor.values(), positive, negative)
#   break

# pbar_train = tqdm(loaders)
# for batch in pbar_train:
#     tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
#     tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
#     tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
#     tf.print(batch[1]['mask'][1, 297: 303])
#     tf.print(batch[1]['times'][1, 297: 303,:])
#     break

In [ ]:
strategy = tf.distribute.get_strategy()
distributed_dataset = strategy.experimental_distribute_dataset(loaders)
pbar_train = tqdm(distributed_dataset)
for batch in pbar_train:
    tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
    tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
    tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
    tf.print(batch[1]['mask'][1, 297: 303], batch[0]['mask'][1, 297: 303], batch[2]['mask'][1, 297: 303])
    tf.print(batch[1]['times'][1, 297: 303,:] , batch[0]['times'][1, 297: 303,:], batch[2]['times'][1, 297: 303,:])
    break

In [ ]:
for anchor, positive, negative in loaders:
  print(anchor, positive, negative)
  break

### Generate Time-series embeddings for dart

In [ ]:
# Define dimensions
d_model = 512

In [ ]:
emb = AstraEmbedding(d_model=d_model, base=10000, rate=0.1, use_band_info=True, use_drop=True, mjd=True)
tensor = emb(negative['input'], negative['times'], negative['band_info'])
print(tensor.shape)
print(tensor)

In [ ]:
from cdshealpix import healpix_to_skycoord

skycoord= healpix_to_skycoord(1525403389526329698, depth=29)
print(skycoord)